# 00 — Clean Prowess Raw Data

In [ ]:
import pandas as pd

RAW_DIR  = 'data/raw'
PROC_DIR = 'data/processed'

RAW_PATH          = f'{RAW_DIR}/prowess_raw_data.xlsx'
MATCHED_SEED_PATH = f'{RAW_DIR}/matched_companies_seed.xlsx'   # manually-curated mapping, read-only

OUTPUT_PATH  = f'{PROC_DIR}/prowess_clean.xlsx'       # clean output – raw file is never touched
MATCHED_PATH = f'{PROC_DIR}/matched_companies.xlsx'   # enriched output – seed is never touched

### Step 1: Build mapping from Prowess Name → Company Code

In [2]:
# Use any data sheet to extract unique Company Code / Company Name pairs
df_ref = pd.read_excel(RAW_PATH, sheet_name='Auditors')

prowess_name_to_code = (
    df_ref[['Company Code', 'Company Name']]
    .drop_duplicates()
    .set_index('Company Name')['Company Code']
    .to_dict()
)

print(f'Unique companies in raw data: {len(prowess_name_to_code)}')

Unique companies in raw data: 500


### Step 2: Populate Prowess Code in matched_companies.xlsx

In [ ]:
matched = pd.read_excel(MATCHED_SEED_PATH)

# Map Prowess Name -> Company Code from raw data
matched['Prowess Code'] = matched['Prowess Name'].map(prowess_name_to_code)

# Store as nullable integer (no decimal .0)
matched['Prowess Code'] = pd.to_numeric(matched['Prowess Code'], errors='coerce').astype('Int64')

print(f"Prowess Code populated : {matched['Prowess Code'].notna().sum()} / {len(matched)}")
print(f"No match (BSE only)    : {matched['Prowess Code'].isna().sum()}")
matched[['BSE Name', 'BSE Code', 'Prowess Name', 'Prowess Code']].head(10)

In [ ]:
# Write the enriched mapping to processed/ (the raw seed is never modified)
with pd.ExcelWriter(MATCHED_PATH, engine='openpyxl') as writer:
    matched.to_excel(writer, index=False)

print(f'Saved {MATCHED_PATH}')

### Step 3: Rename Company Name → BSE Name in all sheets of prowess_raw_data.xlsx

In [5]:
# Build Company Code -> BSE Name mapping
code_to_bse = {
    prowess_name_to_code[p_name]: bse_name
    for _, row in matched.iterrows()
    for p_name, bse_name in [(row['Prowess Name'], row['BSE Name'])]
    if p_name and p_name != '-' and p_name in prowess_name_to_code
}

print(f'Code -> BSE Name entries: {len(code_to_bse)}')

Code -> BSE Name entries: 233


In [6]:
xl = pd.ExcelFile(RAW_PATH)
sheet_names = xl.sheet_names

# Read all sheets into memory
all_sheets = {
    sheet: pd.read_excel(RAW_PATH, sheet_name=sheet)
    for sheet in sheet_names
}

print(f'Loaded {len(all_sheets)} sheets: {sheet_names}')

Loaded 18 sheets: ['Index', 'Auditors', 'Bankers', 'Board_Committees_Meetings', 'Board_Meeting_Dates_Purpose', 'Board_of_Directors_Remuneration', 'Equity_Ownership', 'Insider_Trading_Transactions', 'Major_Owners_of_Equity_Shares', 'Outstanding_Balances', 'Related_Parties_Names', 'Revenue_Expense_Transactions', 'Shares_Pledged_by_Major_Owners', 'Shares_Pledged_by_Promoters', 'Subsidiaries', 'Transactions_Totals_by_Party_Ty', 'Trend_in_Equity_Ownership', 'Trends_in_Directors_Remuneratio']


In [7]:
# Rename Company Name in every sheet that has both Company Code and Company Name
rename_summary = {}

for sheet, df in all_sheets.items():
    if 'Company Name' in df.columns and 'Company Code' in df.columns:
        original = df['Company Name'].copy()
        df['Company Name'] = df['Company Code'].map(code_to_bse).fillna(df['Company Name'])
        rename_summary[sheet] = (original != df['Company Name']).sum()
        all_sheets[sheet] = df

print('Rows renamed per sheet:')
for sheet, count in rename_summary.items():
    print(f'  {sheet}: {count}')

Rows renamed per sheet:
  Auditors: 2669
  Bankers: 3655
  Board_Committees_Meetings: 31011
  Board_Meeting_Dates_Purpose: 12020
  Board_of_Directors_Remuneration: 7692
  Equity_Ownership: 8547
  Insider_Trading_Transactions: 10201
  Major_Owners_of_Equity_Shares: 8765
  Outstanding_Balances: 5725
  Related_Parties_Names: 2
  Revenue_Expense_Transactions: 9499
  Shares_Pledged_by_Major_Owners: 409
  Shares_Pledged_by_Promoters: 1751
  Subsidiaries: 7827
  Transactions_Totals_by_Party_Ty: 26303
  Trend_in_Equity_Ownership: 9194
  Trends_in_Directors_Remuneratio: 5666


In [8]:
# Write all sheets to the new prowess_data.xlsx (raw file is never touched)
with pd.ExcelWriter(OUTPUT_PATH, engine='openpyxl') as writer:
    for sheet, df in all_sheets.items():
        df.to_excel(writer, sheet_name=sheet, index=False)

### Verification

In [9]:
# Spot-check: first 5 rows of Auditors sheet from the new file
df_check = pd.read_excel(OUTPUT_PATH, sheet_name='Auditors', nrows=5)
df_check[['Company Code', 'Company Name']]

,Company Code,Company Name
0,384392,360 ONE WAM LIMITED
1,384392,360 ONE WAM LIMITED
2,384392,360 ONE WAM LIMITED
3,384392,360 ONE WAM LIMITED
4,384392,360 ONE WAM LIMITED
